# 🌊 SST Amérique Centrale 2024–2025
### Données Copernicus Marine — OSTIA L4 NRT

**Dataset** : `METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2`  
**Région** : Amérique Centrale + Caraïbes + Pacifique Est tropical  
**Résolution** : 0.05° × 0.05° (≈ 5 km), journalière  

---
> **Prérequis** : compte gratuit sur [marine.copernicus.eu](https://marine.copernicus.eu)  
> **Connexion** : lancer une fois en terminal `copernicusmarine login`


## 0. Installation des dépendances

In [ ]:
# À lancer une seule fois
import sys
!micromamba install -c conda-forge --yes \
    copernicusmarine xarray matplotlib cartopy cmocean numpy 2>/dev/null | tail -3
print("✓ Dépendances OK")

## 1. Imports

In [ ]:
import copernicusmarine
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cmocean
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path

OUT_DIR = Path("output_sst")
OUT_DIR.mkdir(exist_ok=True)

print("✓ Imports OK")

## 2. Paramètres

In [ ]:
# ── Région Amérique Centrale ──────────────────────────────────────
LON_MIN, LON_MAX = -100, -60
LAT_MIN, LAT_MAX =    5,  30

# ── Dataset Copernicus ─────────────────────────────────────────────
#DATASET_ID = "METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2"
DATASET_ID = "METOFFICE-GLO-SST-L4-REP-OBS-SST"

VARIABLE   = "analysed_sst"

print(f"Région  : lon [{LON_MIN}, {LON_MAX}]  lat [{LAT_MIN}, {LAT_MAX}]")
print(f"Dataset : {DATASET_ID}")

## 3. Téléchargement des données
Les données sont chargées en mode **lazy** (streaming via Zarr) — rien n'est téléchargé jusqu'aux calculs.

In [ ]:
ds = copernicusmarine.open_dataset(
    dataset_id        = DATASET_ID,
    variables         = [VARIABLE],
    minimum_longitude = LON_MIN,
    maximum_longitude = LON_MAX,
    minimum_latitude  = LAT_MIN,
    maximum_latitude  = LAT_MAX,
    start_datetime    = "2024-01-01",
    end_datetime      = "2024-12-31",
)

# Kelvin → Celsius
sst = ds[VARIABLE] - 273.15
sst.attrs["units"]     = "°C"
sst.attrs["long_name"] = "Sea Surface Temperature"

print(ds)
print(f"\nDimensions : {dict(sst.sizes)}")

## 4. Calcul des moyennes mensuelles

In [ ]:
sst_monthly = sst.resample(time="ME").mean()
print(f"Nombre de mois : {len(sst_monthly.time)}")
sst_monthly

## 5. Fonction de cartographie

In [ ]:
def base_map(ax, title=""):
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor="#d4c9a8", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle="--", zorder=3)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                      color="gray", alpha=0.5, linestyle="--")
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlocator = mticker.FixedLocator(range(-100, -55, 10))
    gl.ylocator = mticker.FixedLocator(range(5, 35, 5))
    if title:
        ax.set_title(title, fontsize=10, pad=6)
    return ax

print("✓ Fonction base_map définie")

## 6. Cartes mensuelles 2024

In [ ]:
year = 2024
sst_yr = sst_monthly.sel(time=sst_monthly.time.dt.year == year)
n_months = len(sst_yr.time)

ncols = 4
nrows = int(np.ceil(n_months / ncols))
vmin, vmax = float(sst_yr.min()), float(sst_yr.max())

fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 4, nrows * 3.2),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

for i, t in enumerate(sst_yr.time.values):
    im = sst_yr.isel(time=i).plot(
        ax=axes[i], transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax,
        add_colorbar=False, add_labels=False,
    )
    base_map(axes[i], title=str(t)[:7])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

cbar = fig.colorbar(im, ax=axes[:n_months], orientation="vertical", shrink=0.6, pad=0.02)
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle(f"SST mensuelle — Amérique Centrale {year}", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / f"sst_monthly_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Cartes mensuelles 2025

In [ ]:
year = 2025
sst_yr = sst_monthly.sel(time=sst_monthly.time.dt.year == year)
n_months = len(sst_yr.time)

ncols = 4
nrows = int(np.ceil(n_months / ncols))
vmin, vmax = float(sst_yr.min()), float(sst_yr.max())

fig, axes = plt.subplots(
    nrows, ncols, figsize=(ncols * 4, nrows * 3.2),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
axes = axes.flatten()

for i, t in enumerate(sst_yr.time.values):
    im = sst_yr.isel(time=i).plot(
        ax=axes[i], transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=vmin, vmax=vmax,
        add_colorbar=False, add_labels=False,
    )
    base_map(axes[i], title=str(t)[:7])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

cbar = fig.colorbar(im, ax=axes[:n_months], orientation="vertical", shrink=0.6, pad=0.02)
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle(f"SST mensuelle — Amérique Centrale {year}", fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / f"sst_monthly_{year}.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Comparaison annuelle 2024 vs 2025

In [ ]:
annual_means = {}

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)

for ax, year in zip(axes, [2024, 2025]):
    sst_yr = sst.sel(time=sst.time.dt.year == year).mean("time")
    annual_means[year] = sst_yr
    im = sst_yr.plot(
        ax=ax, transform=ccrs.PlateCarree(),
        cmap=cmocean.cm.thermal, vmin=24, vmax=32,
        add_colorbar=False, add_labels=False,
    )
    base_map(ax, title=f"Moyenne annuelle {year}")

cbar = fig.colorbar(im, ax=axes, orientation="vertical", shrink=0.8, pad=0.04)
cbar.set_label("SST (°C)", fontsize=11)
fig.suptitle("SST annuelle — Amérique Centrale", fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR / "sst_annual_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Anomalie SST (2025 − 2024)

In [ ]:
anomalie = annual_means[2025] - annual_means[2024]
lim = float(max(abs(anomalie.min()), abs(anomalie.max())))

fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
im = anomalie.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap="RdBu_r", vmin=-lim, vmax=lim,
    add_colorbar=False, add_labels=False,
)
base_map(ax, title="Anomalie SST (2025 − 2024)")
cbar = fig.colorbar(im, ax=ax, orientation="vertical", shrink=0.8, pad=0.04)
cbar.set_label("ΔT (°C)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "sst_anomaly_2025_minus_2024.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Série temporelle (moyenne spatiale)

In [ ]:
sst_ts = sst.mean(dim=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(14, 4))
for year, color, lw in [(2024, "#2166ac", 1.4), (2025, "#d6604d", 1.4)]:
    ts = sst_ts.sel(time=sst_ts.time.dt.year == year)
    ax.plot(ts.time.values, ts.values, label=str(year), color=color, linewidth=lw)

ax.set_ylabel("SST moyenne (°C)", fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.set_title("Série temporelle SST — Amérique Centrale (moyenne spatiale)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "sst_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Fichiers générés

In [ ]:
for f in sorted(OUT_DIR.glob("*.png")):
    print(f"✓ {f}")